# MapleStory Job Category Classifier, Ollama Edition

This notebook upgrades the Day 1 MapleStory classifier to use a local open-source model running through Ollama.


**Workflow**

Website → Retrieve MapleStory class information → Build prompt → Llama 3.2 via Ollama → Classify MapleStory job

**Benefits**
- No API costs
- Runs locally
- Same OpenAI-compatible interface

## Install / Import Dependencies

In [ ]:
import requests
from bs4 import BeautifulSoup
from openai import OpenAI

## Verify Ollama

In [ ]:
response = requests.get("http://localhost:11434")

print(response.text)

## Create Ollama Client

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

client = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"
)

## Quick Smoke Test

In [ ]:
response = client.chat.completions.create(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": "Tell me a fun fact"
        }
    ]
)

print(response.choices[0].message.content)

## Website Scraper

In [ ]:
def fetch_website_contents(url):

    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=30
    )

    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    return soup.get_text(
        "\n",
        strip=True
    )

## Download MapleStory Data

In [ ]:
url = "https://www.digitaltq.com/maplestory-classes-jobs"

maplestory = fetch_website_contents(url)

print(
    f"Website text length: {len(maplestory):,}"
)

## Select Job

In [ ]:
job_name = "Kain"

print(
    f"Searching for: {job_name}"
)

print(
    f"Found in text? {job_name in maplestory}"
)

## Prompt Engineering

In [ ]:
system_prompt = """
You are a MapleStory classifier.

Use ONLY the website text provided.

Rules:

1. Find the exact job name.
2. Quote the exact evidence.
3. Identify the nearest category heading.
4. If not found, return UNKNOWN.
5. Do not use prior MapleStory knowledge.
6. Do not assume missing information.

Return:

Evidence:
...

Category:
...
"""

## User Prompt

In [ ]:
user_prompt = f"""
Website information:

{maplestory}

Question:

Which category contains {job_name}?

First explain how you found it.

Then provide the category.
"""

## Build Messages

In [ ]:
messages = [
    {
        "role": "system",
        "content": system_prompt
    },
    {
        "role": "user",
        "content": user_prompt
    }
]

## Run Classification

In [ ]:
response = client.chat.completions.create(
    model="llama3.2",
    messages=messages,
    temperature=0
)

print(
    response.choices[0].message.content
)

## Test Multiple Jobs

In [ ]:
jobs = [
    "Kain",
    "Night Lord",
    "Angelic Buster",
    "Cadena",
    "Hero"
]

for job in jobs:

    prompt = f"""
    Website information:

    {maplestory}

    Which category contains {job}?
    """

    response = client.chat.completions.create(
        model="llama3.2",
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    print("\n" + "=" * 50)
    print(job)
    print("=" * 50)

    print(
        response.choices[0].message.content
    )

## Reflection

Compared with GPT-4.1-mini:

**Advantages**
・Free
・Private
・Runs locally

**Disadvantages**
・Slower on large prompts
・Lower reasoning quality
・More likely to miss evidence

**Key Learning:**
The retrieval pipeline, prompts, and business logic remained unchanged.

Only the model provider changed:

GPT-4.1-mini 
↓
Llama 3.2 via Ollama

This demonstrates the value of OpenAI-compatible APIs.